# Gate.io Perp — CcxtStorage tests
Tests OHLC, funding payment history, and symbol mapping for GATEIO.F via `CcxtStorage`.

**Fix verified:** `_get_or_build_symbol_map` is now keyed by `(exchange, market)` and filters
by CCXT market type, preventing Gate.io options from colliding with perpetual swaps.

In [1]:
import pandas as pd
from qubx.data.storages.ccxt import CcxtStorage

storage = CcxtStorage()
reader = storage.get_reader("GATEIO.F", "SWAP")

start = pd.Timestamp("2026-02-24")
stop  = pd.Timestamp("2026-03-03")

In [2]:
# Verify fix: BTCUSDT should now resolve to the perp, not an option
sym_map = storage._get_or_build_symbol_map("GATEIO.F", "SWAP")
ccxt_sym, instr = sym_map["BTCUSDT"]
print(f"BTCUSDT -> ccxt: {ccxt_sym!r}  instr: {instr}  is_futures: {instr.is_futures()}")

# Confirm no options in the swap sym_map
has_options = any(
    storage._exchanges["GATEIO.F"].markets[cs].get("type") == "option"
    for cs, _ in sym_map.values()
)
print(f"Options in SWAP sym_map: {has_options}  (expected: False)")

BTCUSDT -> ccxt: 'BTC/USDT:USDT'  instr: GATEIO.F:SWAP:BTCUSDT  is_futures: True
Options in SWAP sym_map: False  (expected: False)


In [3]:
# OHLC — price should be ~$85k with real volume
ohlc = reader.read("BTCUSDT", "ohlc(1h)", start, stop)
print(f"ohlc(1h) rows: {len(ohlc)}")
ohlc.to_pd().tail()

ohlc(1h) rows: 169


,open,high,low,close,volume,quote_volume
timestamp,,,,,,
2026-03-02 20:00:00,68927.9,69474.5,68541.4,69065.9,48727941.0,3.365439e+12
2026-03-02 21:00:00,69065.8,69498.9,68953.3,69373.6,26803293.0,1.859441e+12
2026-03-02 22:00:00,69369.5,69460.0,69105.1,69406.7,18476292.0,1.282378e+12
2026-03-02 23:00:00,69406.6,69448.3,68704.4,68798.0,31849742.0,2.191199e+12
2026-03-03 00:00:00,68797.9,69134.5,68610.1,69048.0,31664017.0,2.186337e+12


In [7]:
# Funding payment — should return rows now
funding_gate = reader.read("BTCUSDT", "funding_payment", start, stop)
print(f"GATEIO.F funding_payment rows: {len(funding_gate)}")
funding_gate.to_pd()

GATEIO.F funding_payment rows: 22


,funding_rate,funding_interval_hours
timestamp,,
2026-02-24 00:00:00,0.000056,8.0
2026-02-24 08:00:00,0.000032,8.0
2026-02-24 16:00:00,-0.000017,8.0
2026-02-25 00:00:00,-0.000016,8.0
2026-02-25 08:00:00,0.000039,8.0
2026-02-25 16:00:00,-0.000031,8.0
2026-02-26 00:00:00,0.000013,8.0
2026-02-26 08:00:00,0.000030,8.0
2026-02-26 16:00:00,0.000034,8.0


In [5]:
# Compare BINANCE.UM vs GATEIO.F funding side by side
binance_reader = storage.get_reader("BINANCE.UM", "SWAP")
funding_binance = binance_reader.read("BTCUSDT", "funding_payment", start, stop)
print(f"BINANCE.UM funding_payment rows: {len(funding_binance)}")
funding_binance.to_pd()

Total GATEIO.F symbols: 2549

BTC-related symbols (42):
  LTCBTC               -> ccxt: LTC/BTC
  XLMBTC               -> ccxt: XLM/BTC
  GTBTC                -> ccxt: GT/BTC
  BTC5LUSDT            -> ccxt: BTC5L/USDT
  BDXBTC               -> ccxt: BDX/BTC
  BTCUSD1              -> ccxt: BTC/USD1
  BTCUSDT              -> ccxt: BTC/USDT:USDT-260925-85000-P
  BTCUSDC              -> ccxt: BTC/USDC
  PUMPBTCUSDT          -> ccxt: PUMPBTC/USDT
  ETHBTC               -> ccxt: ETH/BTC
  SOLOBTC              -> ccxt: SOLO/BTC
  CRVBTC               -> ccxt: CRV/BTC
  FILBTC               -> ccxt: FIL/BTC
  BTC3SUSDT            -> ccxt: BTC3S/USDT
  IOTABTC              -> ccxt: IOTA/BTC
  WNCGBTC              -> ccxt: WNCG/BTC
  ADABTC               -> ccxt: ADA/BTC
  PBTC35AUSDT          -> ccxt: PBTC35A/USDT
  QTUMBTC              -> ccxt: QTUM/BTC
  GTBTCUSDT            -> ccxt: GTBTC/USDT
  GTBTCBTC             -> ccxt: GTBTC/BTC
  BTCGUSD              -> ccxt: BTC/GUSD
  DOTBTC        